# Practical Machine Learning Notebook: Simple Linear Regression, Multiple Linear Regression, and Logistic Regression

**Dataset:** `Std_data.csv`  
**Notebook file:** `Std_data_regression_practical.ipynb`

## Learning objectives
By the end of this practical, you should be able to:
1. Inspect and prepare a small tabular dataset with pandas and NumPy.
2. Fit and interpret simple and multiple linear regression models.
3. Fit a logistic regression classifier and interpret predicted probabilities.
4. Visualize predictions, residuals, and classification results.
5. Choose evaluation metrics appropriate to regression or classification.
6. Recognize why five observations are far too few to draw reliable conclusions.

> **Data integrity note:** The notebook first looks for `Std_data.csv` in the current working directory. If it is not found, it creates a clearly labelled *illustrative example* with five rows and the requested columns. Those example outcomes are not actual student records. Replace the example with your real file when available.

## 1. Introduction to supervised machine learning

Supervised machine learning uses examples with known outcomes to learn a relationship between input features and a target. In **regression**, the target is numerical. In **classification**, the target is a category.

- **Simple linear regression** uses one feature to predict a numerical target.
- **Multiple linear regression** uses two or more features to predict a numerical target.
- **Logistic regression** estimates the probability of a class, making it a classification method despite the word “regression” in its name.

The three models below are educational demonstrations. With only five records, model estimates and evaluation metrics will be extremely unstable and should not be treated as evidence of real-world predictive performance.

In [ ]:
# Import the libraries used throughout the notebook.
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, roc_curve
)

# Use a consistent visual style.
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

**Interpretation:** The imports provide data handling, plotting, preprocessing pipelines, three estimators, and the appropriate regression/classification metrics. A fixed random seed makes the demonstration split reproducible; it does not make the small dataset more representative.

## 2. Dataset loading and inspection

The required columns are `Student_ID`, `Study_Hours`, `Attendance_Score`, and `Status`. `Student_ID` uniquely identifies a student; it is not a meaningful measurement and should not be used as a predictive feature. Treating strings such as `std0001` as numbers would create an arbitrary ordering rather than a useful student characteristic.

In [ ]:
# Load the real CSV if present. Otherwise, create an explicitly labelled example dataset.
data_path = Path("Std_data.csv")
USING_EXAMPLE_DATA = not data_path.exists()

if not USING_EXAMPLE_DATA:
    df = pd.read_csv(data_path)
    print(f"Loaded dataset from: {data_path.resolve()}")
    print("These rows are read from the supplied CSV file.")
else:
    # These values are invented only for demonstrating the workflow; they are not actual student data.
    df = pd.DataFrame({
        "Student_ID": ["std0001", "std0002", "std0003", "std0004", "std0005"],
        "Study_Hours": [1, 2, 3, 4, 5],
        "Attendance_Score": [4, 6, 8, 10, 12],
        "Status": ["Fail", "Fail", "Pass", "Pass", "Pass"]
    })
    print("WARNING: Std_data.csv was not found.")
    print("Using clearly labelled ILLUSTRATIVE EXAMPLE DATA, not actual student records.")

# Confirm that all required columns are present.
required_columns = ["Student_ID", "Study_Hours", "Attendance_Score", "Status"]
missing_columns = [col for col in required_columns if col not in df.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

# Display the records and basic structure.
display(df)
print("Shape (rows, columns):", df.shape)
print("\nData types:")
print(df.dtypes)
print("\nDescriptive statistics (including categorical columns):")
display(df.describe(include="all").T)
print("\nMissing values per column:")
display(df.isna().sum().to_frame("missing_count"))
print("\nExact duplicate rows:", int(df.duplicated().sum()))
print("Duplicate Student_ID values:", int(df["Student_ID"].duplicated().sum()))

**Interpretation:** Review the displayed values before modelling. The shape tells you the number of records and fields; data types indicate how pandas interprets each column; descriptive statistics summarize numerical distributions and category counts. Missing values and duplicate records are flagged, not automatically removed. If the fallback example is in use, every result in this notebook is based on illustrative values.

## 3. Data cleaning and preparation

Cleaning makes data types and categories consistent while preserving the source records. This notebook:
- converts the two score columns to numeric values (unparseable values become missing and are reported);
- trims whitespace and normalizes capitalization in `Status`;
- accepts `Pass` and `Fail` after normalization and flags unexpected labels;
- checks the stated score ranges: `Study_Hours` from 1 to 5 and `Attendance_Score` from 0 to 14;
- reports duplicates and missing data rather than silently deleting rows.

Only records with usable values for a specific model can be used by that model. The reason for any exclusion is printed. If data needs correction, verify it against the source rather than guessing.

In [ ]:
# Work on a copy so the original loaded table remains available for comparison.
data = df.copy()

# Convert scores to numeric; invalid text becomes NaN and is reported below.
for col in ["Study_Hours", "Attendance_Score"]:
    data[col] = pd.to_numeric(data[col], errors="coerce")

# Standardize status labels, while leaving missing labels missing.
data["Status"] = data["Status"].astype("string").str.strip().str.title()
data["Status"] = data["Status"].replace({"": pd.NA, "Nan": pd.NA, "None": pd.NA})

# Report unexpected categories rather than silently mapping them to a known class.
unexpected_status = sorted(set(data["Status"].dropna().unique()) - {"Pass", "Fail"})
print("Unexpected Status labels after trimming/capitalization:", unexpected_status)

# Report range violations. Do not clip or change the observed values automatically.
study_out_of_range = data["Study_Hours"].notna() & ~data["Study_Hours"].between(1, 5)
attendance_out_of_range = data["Attendance_Score"].notna() & ~data["Attendance_Score"].between(0, 14)
print("Study_Hours outside [1, 5]:", int(study_out_of_range.sum()))
print("Attendance_Score outside [0, 14]:", int(attendance_out_of_range.sum()))

# Report current missing values and duplicate rows after type/category cleaning.
print("\nMissing values after conversion:")
display(data.isna().sum().to_frame("missing_count"))
print("Duplicate rows:", int(data.duplicated().sum()))
print("Duplicate Student_ID values:", int(data["Student_ID"].duplicated().sum()))

# Create a modelling table with valid score ranges and known target categories.
# Exclusion is explicit: these rows cannot support the specified analyses as-is.
valid_ranges = (
    data["Study_Hours"].between(1, 5) &
    data["Attendance_Score"].between(0, 14)
)
valid_status = data["Status"].isin(["Pass", "Fail"])
model_data = data.loc[valid_ranges & valid_status].copy()

print(f"Rows in source data: {len(data)}")
print(f"Rows with valid scores and Pass/Fail status: {len(model_data)}")
if len(model_data) != len(data):
    print("Rows excluded from the common modelling table:")
    display(data.loc[~(valid_ranges & valid_status)])

# Encode the classification target: Pass=1 and Fail=0.
model_data["Status_Binary"] = model_data["Status"].map({"Pass": 1, "Fail": 0}).astype(int)
display(model_data)

**Interpretation:** The printed checks show whether conversion or category cleanup revealed problems. Scores outside the documented ranges are not silently corrected. The common modelling table includes rows with valid score ranges and a known Pass/Fail label so that all three demonstrations can use a consistent base. For a real project, investigate excluded rows and document corrections from an authoritative source. `Student_ID` is retained for display but not used as a feature.

## 4. Exploratory data analysis

Exploratory analysis helps reveal the scale, distribution, and possible relationships in the available variables. With five observations, a pattern can be driven by one record and should not be generalized to a wider student population.

In [ ]:
# Show summary statistics for the prepared modelling data.
display(model_data[["Study_Hours", "Attendance_Score"]].describe())

# Plot distributions and the relationship between the two numerical variables.
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.histplot(data=model_data, x="Study_Hours", bins=5, ax=axes[0])
axes[0].set_title("Study Hours Distribution")
sns.histplot(data=model_data, x="Attendance_Score", bins=5, ax=axes[1])
axes[1].set_title("Attendance Score Distribution")
sns.scatterplot(
    data=model_data, x="Study_Hours", y="Attendance_Score",
    hue="Status", style="Status", s=100, ax=axes[2]
)
axes[2].set_title("Study Hours vs Attendance")
plt.tight_layout()
plt.show()

# Display class counts, including only the standardized categories.
display(model_data["Status"].value_counts().reindex(["Fail", "Pass"], fill_value=0).to_frame("count"))

**Interpretation:** Histograms show the observed spread; the scatter plot shows whether the two numerical variables appear associated in these records. Class counts reveal whether Pass and Fail are imbalanced. Neither a visual trend nor a class count establishes causation or reliable population-level behaviour.

## 5. Model One — Simple Linear Regression

### Concept

Simple linear regression estimates a straight-line relationship between one numerical predictor and one numerical target. Here, the independent variable is `Study_Hours` (\(X\)) and the dependent variable is `Attendance_Score` (\(y\)). This is an educational modelling choice because both columns are numeric; it does **not** establish that study hours cause better attendance.

The model estimates:

\[
\hat{y} = b_0 + b_1X
\]

where \(b_0\) is the intercept and \(b_1\) is the coefficient (slope). When there are enough usable records, we use a train/test split. With five rows, the test set will contain very few observations, so the evaluation is only a workflow demonstration.

In [ ]:
# Select only the required feature and target; never include Student_ID.
simple_data = model_data[["Study_Hours", "Attendance_Score"]].dropna()
X_simple = simple_data[["Study_Hours"]]
y_simple = simple_data["Attendance_Score"]

simple_model = None
simple_predictions = None
simple_actual = None
simple_train_predictions = None
simple_metrics = None

# A small holdout split is possible with at least three complete rows.
if len(simple_data) >= 3:
    Xs_train, Xs_test, ys_train, ys_test = train_test_split(
        X_simple, y_simple, test_size=0.4, random_state=RANDOM_STATE
    )
    simple_model = LinearRegression()
    simple_model.fit(Xs_train, ys_train)
    simple_predictions = simple_model.predict(Xs_test)
    simple_actual = ys_test.to_numpy()
    simple_train_predictions = simple_model.predict(Xs_train)

    print("Intercept (b0):", round(float(simple_model.intercept_), 4))
    print("Study_Hours coefficient (b1):", round(float(simple_model.coef_[0]), 4))
    print(f"Estimated equation: Attendance_Score = {simple_model.intercept_:.3f} + ({simple_model.coef_[0]:.3f} × Study_Hours)")
    display(pd.DataFrame({
        "Study_Hours": Xs_test["Study_Hours"].to_numpy(),
        "Actual_Attendance_Score": simple_actual,
        "Predicted_Attendance_Score": simple_predictions
    }).reset_index(drop=True))
else:
    print("Not enough complete rows for a train/test demonstration; at least 3 are required.")

**Interpretation:** The intercept is the fitted attendance score when study hours equal zero, which is outside the stated study-hours range and may not have a practical meaning here. The slope estimates the change in predicted attendance score for a one-unit increase in study hours within the fitted linear model. It is an association in this dataset, not a causal effect. Predictions are estimates and may fall outside the target's allowed range because ordinary linear regression is not constrained to 0–14.

In [ ]:
# Plot the data and the fitted line using the model trained on the training subset.
if simple_model is not None:
    x_line = np.linspace(X_simple["Study_Hours"].min(), X_simple["Study_Hours"].max(), 100)
    y_line = simple_model.predict(pd.DataFrame({"Study_Hours": x_line}))

    plt.figure(figsize=(7, 5))
    sns.scatterplot(x=X_simple["Study_Hours"], y=y_simple, s=90, label="Observed records")
    plt.plot(x_line, y_line, color="red", label="Fitted line (training data)")
    plt.xlabel("Study_Hours")
    plt.ylabel("Attendance_Score")
    plt.title("Simple Linear Regression: Fitted Line")
    plt.legend()
    plt.show()

    # Actual-versus-predicted plot; the diagonal indicates perfect predictions.
    plt.figure(figsize=(6, 5))
    plt.scatter(simple_actual, simple_predictions, s=90)
    low = min(np.min(simple_actual), np.min(simple_predictions))
    high = max(np.max(simple_actual), np.max(simple_predictions))
    plt.plot([low, high], [low, high], "r--", label="Perfect prediction (y = x)")
    plt.xlabel("Actual attendance score")
    plt.ylabel("Predicted attendance score")
    plt.title("Simple Regression: Actual vs Predicted")
    plt.legend()
    plt.show()

    # Residual = actual - predicted. Residuals above zero mean underprediction.
    simple_residuals = simple_actual - simple_predictions
    plt.figure(figsize=(6, 5))
    plt.scatter(simple_predictions, simple_residuals, s=90)
    plt.axhline(0, color="red", linestyle="--")
    plt.xlabel("Predicted attendance score")
    plt.ylabel("Residual (actual - predicted)")
    plt.title("Simple Regression: Residuals vs Predicted")
    plt.show()

    # Explicit residual-versus-fitted view, using the same fitted values.
    plt.figure(figsize=(6, 5))
    plt.scatter(simple_predictions, simple_residuals, s=90)
    plt.axhline(0, color="black", linestyle="--")
    plt.xlabel("Fitted / predicted attendance score")
    plt.ylabel("Residual")
    plt.title("Residual-versus-Fitted Plot")
    plt.show()
else:
    print("Plots skipped because no simple regression model was fitted.")

**How to read these plots:** Points near the diagonal in the actual-versus-predicted plot indicate closer predictions. Residuals are actual minus predicted: positive values mean the model underpredicted and negative values mean it overpredicted. A residual plot with random scatter around zero is generally preferable to a clear curve or funnel, but a handful of points cannot reliably diagnose model assumptions. The fitted line is based on the training subset, not the full dataset.

## 6. Model Two — Multiple Linear Regression

### Concept and feature design

Multiple linear regression predicts one numerical target from two or more predictors. For this demonstration, the target remains `Attendance_Score`, and the predictors are `Study_Hours` and categorical `Status`.

A categorical label cannot be meaningfully treated as a numeric quantity, so `Status` is one-hot encoded into indicator columns. A `ColumnTransformer` and `Pipeline` keep preprocessing and modelling together.

> **Important limitation:** This feature design may be circular or involve target leakage in a real setting if student status is assigned using attendance or related information. It is included to demonstrate categorical encoding and multiple regression, not as a recommended operational prediction design. No extra predictor is fabricated, and `Student_ID` is excluded.

In [ ]:
# Define predictors and target. Status is categorical and will be one-hot encoded.
multiple_data = model_data[["Study_Hours", "Status", "Attendance_Score"]].dropna()
X_multiple = multiple_data[["Study_Hours", "Status"]]
y_multiple = multiple_data["Attendance_Score"]

multiple_model = None
multiple_predictions = None
multiple_actual = None
multiple_feature_names = None
multiple_metrics = None

if len(multiple_data) >= 3 and multiple_data["Status"].nunique() >= 1:
    Xm_train, Xm_test, ym_train, ym_test = train_test_split(
        X_multiple, y_multiple, test_size=0.4, random_state=RANDOM_STATE
    )

    # Numeric study hours pass through unchanged; status becomes indicator columns.
    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", "passthrough", ["Study_Hours"]),
            ("categorical", OneHotEncoder(handle_unknown="ignore"), ["Status"])
        ],
        remainder="drop"
    )
    multiple_model = Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", LinearRegression())
    ])
    multiple_model.fit(Xm_train, ym_train)
    multiple_predictions = multiple_model.predict(Xm_test)
    multiple_actual = ym_test.to_numpy()

    # Obtain the transformed feature names and matching regression coefficients.
    fitted_preprocessor = multiple_model.named_steps["preprocessor"]
    multiple_feature_names = fitted_preprocessor.get_feature_names_out()
    multiple_coefficients = multiple_model.named_steps["regressor"].coef_
    print("Intercept:", round(float(multiple_model.named_steps["regressor"].intercept_), 4))
    print("\nCoefficients:")
    display(pd.DataFrame({
        "Feature": multiple_feature_names,
        "Coefficient": multiple_coefficients
    }))
    display(pd.DataFrame({
        "Study_Hours": Xm_test["Study_Hours"].to_numpy(),
        "Status": Xm_test["Status"].to_numpy(),
        "Actual_Attendance_Score": multiple_actual,
        "Predicted_Attendance_Score": multiple_predictions
    }).reset_index(drop=True))
else:
    print("Insufficient complete records or feature variation for the multiple regression demonstration.")

**Interpretation:** The `Study_Hours` coefficient estimates the modelled change in attendance score for a one-unit increase in study hours while holding the encoded status features fixed. A status coefficient compares a category with the omitted reference category, conditional on the other features and the model's encoding. With very few rows, coefficients may be unstable or not separately identifiable; interpret them as fitted mathematical associations, not causal effects.

In [ ]:
# Plot actual versus predicted values and residuals for multiple regression.
if multiple_model is not None:
    plt.figure(figsize=(6, 5))
    plt.scatter(multiple_actual, multiple_predictions, s=90)
    low = min(np.min(multiple_actual), np.min(multiple_predictions))
    high = max(np.max(multiple_actual), np.max(multiple_predictions))
    plt.plot([low, high], [low, high], "r--", label="Perfect prediction (y = x)")
    plt.xlabel("Actual attendance score")
    plt.ylabel("Predicted attendance score")
    plt.title("Multiple Regression: Actual vs Predicted")
    plt.legend()
    plt.show()

    multiple_residuals = multiple_actual - multiple_predictions
    plt.figure(figsize=(6, 5))
    plt.scatter(multiple_predictions, multiple_residuals, s=90)
    plt.axhline(0, color="red", linestyle="--")
    plt.xlabel("Predicted attendance score")
    plt.ylabel("Residual (actual - predicted)")
    plt.title("Multiple Regression: Residuals vs Predicted")
    plt.show()

    # Coefficient plot helps compare the fitted weights of encoded predictors.
    coef_df = pd.DataFrame({
        "Feature": multiple_feature_names,
        "Coefficient": multiple_model.named_steps["regressor"].coef_
    }).sort_values("Coefficient")
    plt.figure(figsize=(8, 4))
    sns.barplot(data=coef_df, x="Coefficient", y="Feature")
    plt.axvline(0, color="black", linestyle="--")
    plt.title("Multiple Regression: Estimated Coefficients")
    plt.tight_layout()
    plt.show()
else:
    print("Multiple regression plots skipped because no model was fitted.")

**How to read these plots:** The diagonal in the actual-versus-predicted plot represents perfect prediction. The residual plot checks whether prediction errors show obvious patterns. The coefficient plot shows the direction and magnitude of fitted weights, but coefficient magnitudes are not always directly comparable across differently scaled features. With a tiny dataset and a potentially circular `Status` predictor, these plots are illustrative rather than evidence of a useful forecasting system.

## 7. Model Three — Logistic Regression

### Concept

Logistic regression is a **classification** algorithm. It predicts the probability of a category and then assigns a class using a decision threshold (commonly 0.5). Here, `Pass` is encoded as 1 and `Fail` as 0; predictors are `Study_Hours` and `Attendance_Score`.

The sigmoid function maps a linear score \(z\) to a probability:

\[
P(y=1\mid X)=\frac{1}{1+e^{-z}}
\]

Here, \(z\) is a linear combination of predictor values and learned coefficients. Numerical features are standardized within a pipeline so they are on comparable scales. Stratification is attempted only when the class counts and sample size permit it.

In [ ]:
# Build a classification table with known labels and valid numerical features.
classification_data = model_data[["Student_ID", "Study_Hours", "Attendance_Score", "Status", "Status_Binary"]].dropna()
X_log = classification_data[["Study_Hours", "Attendance_Score"]]
y_log = classification_data["Status_Binary"]

print("Class counts (0=Fail, 1=Pass):")
display(y_log.value_counts().sort_index().rename(index={0: "Fail (0)", 1: "Pass (1)"}).to_frame("count"))

log_model = None
log_predictions = None
log_probabilities = None
y_log_test = None
X_log_test = None
log_split_note = ""

# A two-class train/test evaluation needs enough examples from both classes.
class_counts = y_log.value_counts()
can_stratify = (
    len(y_log) >= 4 and
    y_log.nunique() == 2 and
    class_counts.min() >= 2
)

if can_stratify:
    try:
        Xl_train, Xl_test, yl_train, yl_test = train_test_split(
            X_log, y_log, test_size=0.4, random_state=RANDOM_STATE, stratify=y_log
        )
        log_split_note = "Stratified train/test split used; test metrics remain highly unstable."
    except ValueError as exc:
        can_stratify = False
        log_split_note = f"Stratified split was not feasible: {exc}"

if not can_stratify:
    # If a reliable class-aware split is not possible, do not fabricate or duplicate records.
    # A small non-stratified split may demonstrate the workflow only if both classes exist.
    if len(y_log) >= 3 and y_log.nunique() == 2:
        Xl_train, Xl_test, yl_train, yl_test = train_test_split(
            X_log, y_log, test_size=0.4, random_state=RANDOM_STATE
        )
        log_split_note = "Non-stratified demonstration split used because class counts were too small for stratification."
    else:
        Xl_train = Xl_test = yl_train = yl_test = None
        log_split_note = "A two-class train/test evaluation is not feasible with the available labels."

print(log_split_note)

if Xl_train is not None and yl_train.nunique() == 2:
    # Scaling is fitted only on training data by the pipeline to avoid leakage from the test set.
    log_model = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(random_state=RANDOM_STATE, max_iter=1000))
    ])
    log_model.fit(Xl_train, yl_train)
    log_predictions = log_model.predict(Xl_test)
    log_probabilities = log_model.predict_proba(Xl_test)[:, 1]
    y_log_test = yl_test.to_numpy()
    X_log_test = Xl_test.copy()

    # Include student IDs for traceability in the displayed evaluation rows, not as model features.
    display_results = classification_data.loc[Xl_test.index, ["Student_ID", "Study_Hours", "Attendance_Score", "Status"]].copy()
    display_results["Actual_Class"] = y_log_test
    display_results["Predicted_Class"] = log_predictions
    display_results["Predicted_Pass_Probability"] = log_probabilities
    display(display_results.reset_index(drop=True))
else:
    print("The classifier was not fitted because the training data did not contain both classes.")

**Interpretation:** The probability column is the model's estimated probability of `Pass`, not a guarantee or a student's ability score. A 0.5 threshold is a convention and may not suit every application. If the training set contains only one class, a standard logistic classifier cannot learn the intended two-class distinction, so the notebook skips fitting rather than manufacturing data.

In [ ]:
# Visualize known classes and, when a model exists, the evaluation results.
plt.figure(figsize=(7, 5))
sns.scatterplot(
    data=classification_data, x="Study_Hours", y="Attendance_Score",
    hue="Status", style="Status", s=110
)
plt.title("Student Records by Status")
plt.show()

if log_model is not None:
    # Confusion matrix uses the explicit label order: Fail=0, Pass=1.
    cm = confusion_matrix(y_log_test, log_predictions, labels=[0, 1])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Fail (0)", "Pass (1)"])
    disp.plot(cmap="Blues", values_format="d")
    plt.title("Logistic Regression: Confusion Matrix")
    plt.show()

    # Class plot: actual and predicted classes are discrete, not continuous regression values.
    positions = np.arange(len(y_log_test))
    plt.figure(figsize=(8, 4))
    plt.scatter(positions - 0.08, y_log_test, label="Actual class", marker="o", s=90)
    plt.scatter(positions + 0.08, log_predictions, label="Predicted class", marker="x", s=90)
    plt.yticks([0, 1], ["Fail (0)", "Pass (1)"])
    plt.xlabel("Evaluation record position")
    plt.ylabel("Class")
    plt.title("Actual vs Predicted Class (not a continuous-value plot)")
    plt.legend()
    plt.show()

    # Show estimated passing probabilities for each evaluation record.
    plt.figure(figsize=(8, 4))
    plt.bar(np.arange(len(log_probabilities)), log_probabilities)
    plt.axhline(0.5, color="red", linestyle="--", label="Default threshold = 0.5")
    plt.ylim(0, 1)
    plt.xlabel("Evaluation record position")
    plt.ylabel("Predicted probability of Pass")
    plt.title("Predicted Pass Probabilities")
    plt.legend()
    plt.show()
else:
    print("Confusion matrix, class, and probability plots skipped because no classifier was fitted.")

**Confusion matrix interpretation:** With labels ordered as Fail (0), Pass (1), the top-left cell is true negatives (actual Fail predicted Fail), the bottom-right is true positives (actual Pass predicted Pass), the top-right is false positives (actual Fail predicted Pass), and the bottom-left is false negatives (actual Pass predicted Fail). The class plot compares discrete labels, while the probability plot shows estimated probabilities. A decision boundary can be misleading with only five points; the optional cell below draws one only when the fitted model and both classes permit it.

In [ ]:
# Draw a decision boundary only if the trained classifier and both training classes are available.
if log_model is not None and len(X_log) >= 3 and y_log.nunique() == 2:
    x_min, x_max = X_log["Study_Hours"].min() - 0.5, X_log["Study_Hours"].max() + 0.5
    y_min, y_max = X_log["Attendance_Score"].min() - 1, X_log["Attendance_Score"].max() + 1
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 150),
        np.linspace(y_min, y_max, 150)
    )
    grid = pd.DataFrame({
        "Study_Hours": xx.ravel(),
        "Attendance_Score": yy.ravel()
    })
    zz = log_model.predict(grid).reshape(xx.shape)

    plt.figure(figsize=(8, 5))
    plt.contourf(xx, yy, zz, alpha=0.2, levels=[-0.5, 0.5, 1.5], cmap="coolwarm")
    sns.scatterplot(
        data=classification_data, x="Study_Hours", y="Attendance_Score",
        hue="Status", style="Status", s=110, edgecolor="black"
    )
    plt.xlabel("Study_Hours")
    plt.ylabel("Attendance_Score")
    plt.title("Illustrative Logistic Regression Decision Regions")
    plt.show()
else:
    print("Decision boundary skipped: a fitted two-class model and suitable feature variation are required.")

**Decision-boundary caution:** The shaded regions show the classes predicted by this fitted model over a grid of feature values. They are not proof of a true boundary between passing and failing. With only five records, the shape can change substantially when even one observation changes.

## 8. Model evaluation and metrics

### Regression metrics

- **MAE (Mean Absolute Error):** average absolute difference between actual and predicted values; lower is better.
- **MSE (Mean Squared Error):** average squared error; it penalizes large errors more heavily and is expressed in squared target units.
- **RMSE (Root Mean Squared Error):** square root of MSE, expressed in the same units as the target.
- **\(R^2\):** compares the model with a baseline that predicts the test-set mean. A negative test \(R^2\) is possible when predictions are worse than that baseline. With very small test sets, \(R^2\) can be especially unstable or undefined.

Classification metrics such as accuracy, precision, recall, and F1 are not appropriate for continuous regression predictions unless a separate, justified conversion to classes is explicitly defined.

In [ ]:
# Compute regression metrics on held-out test rows only, and label them as holdout metrics.
def regression_metric_row(name, actual, predicted):
    return {
        "Model": name,
        "Evaluation data": "Held-out test data",
        "MAE": mean_absolute_error(actual, predicted),
        "MSE": mean_squared_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "R2": r2_score(actual, predicted) if len(actual) >= 2 else np.nan
    }

regression_rows = []
if simple_model is not None and simple_actual is not None:
    regression_rows.append(regression_metric_row("Simple Linear Regression", simple_actual, simple_predictions))
if multiple_model is not None and multiple_actual is not None:
    regression_rows.append(regression_metric_row("Multiple Linear Regression", multiple_actual, multiple_predictions))

regression_metrics = pd.DataFrame(regression_rows)
if not regression_metrics.empty:
    display(regression_metrics.round(4))
else:
    print("No regression test metrics available.")

# These test metrics are descriptive only: the holdout set has very few observations.
print("Reminder: do not treat metrics from a tiny test set as reliable evidence of generalization.")

**Interpretation:** Lower MAE, MSE, and RMSE indicate smaller errors on the evaluated rows. Higher \(R^2\) is not automatically evidence of a good model, especially with a tiny test set. If the test target has fewer than two observations, \(R^2\) is not reported; if target values are constant, it may also be uninformative. Metrics shown here are held-out test metrics, not training metrics.

### Classification metrics

- **Accuracy:** fraction of predictions that are correct.
- **Precision:** among records predicted Pass, the fraction that actually passed.
- **Recall:** among actual Pass records, the fraction correctly identified.
- **F1 score:** harmonic mean of precision and recall.
- **ROC-AUC:** measures ranking of positive versus negative cases across thresholds; it requires both classes in the evaluation labels.

When a class is absent, some metrics may be undefined or misleading. `zero_division=0` makes precision/recall/F1 return zero rather than raising a warning when a denominator is zero; it does not make the result meaningful. Do not report ROC-AUC when the evaluation labels contain only one class.

In [ ]:
# Compute classification metrics only if a fitted model produced held-out predictions.
if log_model is not None and y_log_test is not None and log_predictions is not None:
    class_metrics = {
        "Evaluation data": "Held-out test data",
        "Accuracy": accuracy_score(y_log_test, log_predictions),
        "Precision (Pass=1)": precision_score(y_log_test, log_predictions, pos_label=1, zero_division=0),
        "Recall (Pass=1)": recall_score(y_log_test, log_predictions, pos_label=1, zero_division=0),
        "F1 (Pass=1)": f1_score(y_log_test, log_predictions, pos_label=1, zero_division=0)
    }
    display(pd.DataFrame([class_metrics]).round(4))

    # ROC-AUC is meaningful only when the evaluation labels contain both classes.
    if len(np.unique(y_log_test)) == 2:
        auc = roc_auc_score(y_log_test, log_probabilities)
        print("ROC-AUC (held-out test data):", round(float(auc), 4))
        fpr, tpr, thresholds = roc_curve(y_log_test, log_probabilities)
        plt.figure(figsize=(6, 5))
        plt.plot(fpr, tpr, label=f"ROC curve (AUC = {auc:.3f})")
        plt.plot([0, 1], [0, 1], "k--", label="No-discrimination reference")
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.title("Receiver Operating Characteristic (ROC) Curve")
        plt.legend()
        plt.show()
    else:
        print("ROC-AUC and ROC curve not reported: evaluation labels lack both classes.")
else:
    print("Classification metrics skipped because a two-class classifier was not fitted.")

**Interpretation:** Read each metric alongside the class counts and confusion matrix. A high accuracy score can occur simply because the model predicts the majority class, and with only one or two test records a single prediction can change the score dramatically. ROC-AUC is omitted when the evaluation subset does not contain both classes. None of these values should be presented as reliable evidence that the model generalizes to other students.

## 9. Actual-versus-predicted plots and residual analysis

This section consolidates the diagnostic ideas from the two regression models. Actual-versus-predicted plots compare predictions with observed scores; residual plots show prediction errors. Residuals are defined as \(e_i = y_i - \hat{y}_i\).

A pattern in residuals can suggest non-linearity, unequal error spread, or an influential observation. However, five observations are not enough to diagnose assumptions robustly. These plots should be used as teaching aids, not as a substitute for a larger and more representative dataset.

In [ ]:
# Consolidated diagnostic table makes the evaluated observations easy to review.
diagnostic_tables = []
if simple_model is not None:
    diagnostic_tables.append(pd.DataFrame({
        "Model": "Simple Linear Regression",
        "Actual": simple_actual,
        "Predicted": simple_predictions,
        "Residual": simple_actual - simple_predictions
    }))
if multiple_model is not None:
    diagnostic_tables.append(pd.DataFrame({
        "Model": "Multiple Linear Regression",
        "Actual": multiple_actual,
        "Predicted": multiple_predictions,
        "Residual": multiple_actual - multiple_predictions
    }))

if diagnostic_tables:
    display(pd.concat(diagnostic_tables, ignore_index=True).round(4))
else:
    print("No regression diagnostics available.")

**Interpretation:** Compare residual sizes and signs, but do not select a model solely because it appears better on two or three held-out records. A defensible comparison requires more data, a pre-defined evaluation plan, and ideally repeated validation on representative samples.

## 10. Model comparison

| Feature | Simple Linear Regression | Multiple Linear Regression | Logistic Regression |
|---|---|---|---|
| Main purpose | Predict a continuous value | Predict a continuous value | Predict a category |
| Predictors | One | Two or more | One or more |
| Target in this notebook | `Attendance_Score` | `Attendance_Score` | `Status` |
| Evaluation | MAE, MSE, RMSE, \(R^2\) | MAE, MSE, RMSE, \(R^2\) | Accuracy, Precision, Recall, F1; ROC-AUC where feasible |
| Main visualization | Scatter plot with regression line | Actual-vs-predicted and residual plots | Class scatter plot and confusion matrix |

Simple linear regression uses one predictor; multiple linear regression uses several. Logistic regression estimates class probabilities and predicts a categorical outcome. Multiple regression in this notebook uses `Status` to predict attendance score only to demonstrate encoding. If `Status` is determined by attendance or related information, the design can be circular or leak target-related information. For a real system, select predictors that would genuinely be available at prediction time and are not derived from the target.

In [ ]:
# Build a compact comparison of which models ran and which metrics are available.
comparison_rows = []
for model_name in ["Simple Linear Regression", "Multiple Linear Regression"]:
    match = regression_metrics[regression_metrics["Model"] == model_name] if not regression_metrics.empty else pd.DataFrame()
    comparison_rows.append({
        "Model": model_name,
        "Task": "Regression",
        "Test metrics available": not match.empty,
        "Caution": "Very small holdout; unstable estimates"
    })

comparison_rows.append({
    "Model": "Logistic Regression",
    "Task": "Classification",
    "Test metrics available": log_model is not None and log_predictions is not None,
    "Caution": "Class counts and tiny holdout limit interpretation"
})
display(pd.DataFrame(comparison_rows))

**Interpretation:** The comparison table reports whether a workflow ran, not which model is universally best. Regression and classification metrics answer different questions and should not be ranked against one another. A real comparison should use a larger dataset and an evaluation strategy suitable for its size and class distribution.

## 11. Limitations and responsible interpretation

1. **Only five observations:** A train/test split leaves very few evaluation rows. One record can materially change the coefficients and metrics.
2. **Example data warning:** If `Std_data.csv` was missing, the results come from illustrative values, not actual student records.
3. **No causal claims:** Associations between study hours, attendance, and status do not show that one variable causes another.
4. **Potential leakage/circularity:** Using `Status` to predict attendance may be unsuitable if status depends on attendance or related information.
5. **No reliable generalization claim:** Five observations are insufficient to infer student performance patterns or establish that a model will work for new students.
6. **Data quality matters:** Confirm labels, ranges, missing records, and duplicates against the source. Never invent or duplicate test records to improve a score.
7. **Fair and careful use:** Student outcomes can affect real people. A model like this should not be used for high-stakes decisions without appropriate data, validation, oversight, and consideration of context.

In [ ]:
# Final audit: show whether this run used a real CSV and summarize the usable sample.
print("Source:", "Std_data.csv" if not USING_EXAMPLE_DATA else "ILLUSTRATIVE EXAMPLE DATA (CSV not found)")
print("Source row count:", len(df))
print("Usable modelling row count:", len(model_data))
print("Usable Status counts:")
display(model_data["Status"].value_counts().reindex(["Fail", "Pass"], fill_value=0).to_frame("count"))
print("\nConclusion: five observations are insufficient for reliable conclusions about student performance or model generalization.")

**Interpretation:** This final audit makes the data source and usable sample size explicit. Do not report a high score from five records as proof of effectiveness. A stronger study should collect substantially more representative observations, define the intended prediction time and target clearly, avoid leakage, and evaluate on data not used during model development.

## 12. Practical exercises

1. **Load real data:** Place your verified `Std_data.csv` next to the notebook and rerun every cell. Compare its class counts and missing values with the example.
2. **Inspect labels:** If `Status` contains labels such as `pass ` or `FAIL`, check how the cleaning step standardizes them. Decide how unexpected labels should be verified and handled.
3. **Interpret a slope:** Explain the simple regression coefficient in one sentence, including why it does not imply causation.
4. **Compare errors:** Compare MAE and RMSE. If RMSE is much larger than MAE, what might that suggest about large errors?
5. **Inspect residuals:** Describe any visible pattern in the residual plot, then explain why five observations are insufficient to make a strong diagnosis.
6. **Think about leakage:** Explain why using `Status` to predict attendance might be circular. Propose a legitimate predictor only if it is actually collected and available at prediction time—do not invent data.
7. **Classification threshold:** Explain how changing the probability threshold from 0.5 might affect false positives and false negatives.
8. **Plan better validation:** Describe what additional data and validation design would be needed before using the model to support real decisions.

In [ ]:
# Optional exercise: test how the classification threshold changes predicted classes.
# This uses the same held-out probabilities and does not retrain the model.
if log_model is not None and log_probabilities is not None:
    threshold_rows = []
    for threshold in [0.3, 0.5, 0.7]:
        threshold_predictions = (log_probabilities >= threshold).astype(int)
        threshold_rows.append({
            "Threshold": threshold,
            "Accuracy": accuracy_score(y_log_test, threshold_predictions),
            "Precision (Pass=1)": precision_score(y_log_test, threshold_predictions, pos_label=1, zero_division=0),
            "Recall (Pass=1)": recall_score(y_log_test, threshold_predictions, pos_label=1, zero_division=0),
            "F1 (Pass=1)": f1_score(y_log_test, threshold_predictions, pos_label=1, zero_division=0)
        })
    display(pd.DataFrame(threshold_rows).round(4))
    print("Exercise: with so few test records, threshold comparisons are highly unstable.")
else:
    print("Threshold exercise unavailable because no classifier probabilities were produced.")

**Exercise interpretation:** A lower threshold usually makes the model predict Pass more readily, which can increase recall but may also increase false positives. A higher threshold may reduce false positives while missing more actual Pass cases. With so few records, these changes may reflect only one prediction.

## 13. Conclusion

This notebook demonstrated data inspection, preparation, exploratory visualization, simple linear regression, multiple linear regression, logistic regression, and appropriate evaluation metrics. The models answer different questions: the two linear models estimate a numerical attendance score, while logistic regression predicts Pass/Fail probabilities and classes.

The main lesson is methodological: choose features and targets that fit the real question, prepare categorical and numerical data appropriately, use metrics suited to the task, and interpret plots and scores in context. **Five observations are insufficient for reliable conclusions about student performance or meaningful claims about model generalization.** Replace the example data with verified records and collect a substantially larger, representative dataset before drawing conclusions.